# 擂台加賽 2:Gemini 自己切框

只給 Gemini 整頁,讓它自己找出文字框並抄寫,再拿 PaddleOCR-VL 的框當參考算重疊率(IoU)。

挑的是上一輪框有問題的頁:
- **p011**:シャコーン、傍點(強調點)
- **p014**:時鐘誤判、カッ 被切掉一半
- **p017**、**p020**:プルルル 電話鈴
- **p022**:ポタッ 連續 6 格
- **p023**:對話密集的頁

**執行階段:CPU**。圖片用上一輪存在 Drive `manga-ocr/out/png/` 的原圖。

## 1. 設定

In [ ]:
PROJECT_ID = ""          # 同上一個 notebook
LOCATION   = "global"
MODEL      = ""          # 留空 = 自動挑最新 Flash;想固定就填上一輪印出的名稱

DRIVE_DIR = "/content/drive/MyDrive/manga-ocr/out"
PAGES     = ["p011", "p014", "p017", "p020", "p022", "p023"]
IOU_MATCH = 0.5

PROMPT = '''這是一頁日文漫畫。找出頁面上所有的文字區塊(對話框、旁白、手寫字、狀聲詞),每個對話框或每組狀聲詞算一個區塊。
- box_2d 用 [ymin, xmin, ymax, xmax],數值是 0–1000 的相對座標,框要貼緊文字本身
- text 逐字抄寫,直書由右到左讀;不要振り仮名(小字注音)和傍点;保留口語寫法,不要改寫
- kind 是 "dialogue" 或 "sfx"
- 沒有文字的圖案(例如時鐘、招牌以外的圖)不要框
只輸出 JSON 陣列:[{"box_2d": [...], "text": "...", "kind": "..."}]'''

## 2. 安裝、授權、選模型

In [ ]:
!pip install -q -U google-genai
from google.colab import auth, drive
auth.authenticate_user()
drive.mount("/content/drive")
import re, json
from google import genai
from google.genai import types
assert PROJECT_ID, "先填 PROJECT_ID"
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)
if not MODEL:
    names = [m.name.split("/")[-1] for m in client.models.list()]
    flash = [n for n in names if re.fullmatch(r"gemini-[\d.-]+-flash(-preview.*)?", n)]
    ver = lambda n: [int(x) for x in re.findall(r"\d+", n.split("-flash")[0])]
    MODEL = max(flash, key=lambda n: (ver(n), "preview" not in n))
print("使用:", MODEL)

## 3. Gemini 看整頁

In [ ]:
import glob
from pathlib import Path
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

paddle = json.load(open(f"{DRIVE_DIR}/results.json"))
paddle = {Path(k).stem: v for k, v in paddle.items()}
paths = {Path(p).stem: p for p in glob.glob(f"{DRIVE_DIR}/png/*")}

def run(stem):
    img = open(paths[stem], "rb").read()
    mime = "image/png" if paths[stem].endswith(".png") else "image/jpeg"
    r = client.models.generate_content(
        model=MODEL,
        contents=[types.Part.from_bytes(data=img, mime_type=mime), PROMPT],
        config=types.GenerateContentConfig(temperature=0, response_mime_type="application/json"),
    )
    try:
        out = json.loads(r.text)
    except Exception:
        m = re.search(r"\[.*\]", r.text or "", re.S); out = json.loads(m.group(0)) if m else []
    w, h = Image.open(paths[stem]).size
    boxes = []
    for o in out:
        y1, x1, y2, x2 = o["box_2d"]
        boxes.append({"bbox": [x1 * w / 1000, y1 * h / 1000, x2 * w / 1000, y2 * h / 1000],
                      "text": o.get("text", ""), "kind": o.get("kind", "")})
    return stem, boxes

with ThreadPoolExecutor(6) as ex:
    gem = dict(ex.map(run, PAGES))
json.dump(gem, open(f"{DRIVE_DIR}/gemini_page_boxes.json", "w"), ensure_ascii=False, indent=1)
for s in PAGES:
    print(s, "Paddle", len(paddle[s]), "框 / Gemini", len(gem[s]), "框")

## 4. 和 Paddle 的框比對

In [ ]:
def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / union if union > 0 else 0

summary = []
for s in PAGES:
    P, G = paddle[s], gem[s]
    pairs = sorted(((iou(p["bbox"], g["bbox"]), i, j) for i, p in enumerate(P) for j, g in enumerate(G)), reverse=True)
    used_p, used_g, matches = set(), set(), []
    for v, i, j in pairs:
        if v < 0.1 or i in used_p or j in used_g: continue
        used_p.add(i); used_g.add(j); matches.append((i, j, v))
    good = [m for m in matches if m[2] >= IOU_MATCH]
    for i, j, v in matches:
        G[j]["match"] = i + 1; G[j]["iou"] = round(v, 2)
    summary.append({"page": s, "paddle": len(P), "gemini": len(G), "iou>=0.5": len(good),
                    "mean_iou(matched)": round(sum(m[2] for m in matches) / max(1, len(matches)), 2),
                    "paddle_missed_by_gemini": [i + 1 for i in range(len(P)) if i not in used_p],
                    "gemini_extra": [j + 1 for j in range(len(G)) if j not in used_g]})
import pandas as pd
display(pd.DataFrame(summary))

## 5. 疊圖:綠 = Paddle,紅 = Gemini(標號 = Gemini 的框序)

In [ ]:
import io, base64, html
from PIL import ImageDraw, ImageFont
from IPython.display import HTML, display
font = ImageFont.load_default(size=34)
import os
os.makedirs(f"{DRIVE_DIR}/gemini_overlay", exist_ok=True)
parts = []
for s in PAGES:
    im = Image.open(paths[s]).convert("RGB"); d = ImageDraw.Draw(im)
    for p in paddle[s]:
        d.rectangle(p["bbox"], outline=(0, 170, 0), width=5)
    for j, g in enumerate(gem[s], 1):
        d.rectangle(g["bbox"], outline=(220, 0, 0), width=3)
        d.text((g["bbox"][0], max(0, g["bbox"][1] - 36)), str(j), fill=(220, 0, 0), font=font, stroke_width=3, stroke_fill="white")
    small = im.copy(); small.thumbnail((1000, 1400))
    small.save(f"{DRIVE_DIR}/gemini_overlay/{s}.jpg", quality=85)
    buf = io.BytesIO(); small.save(buf, "JPEG", quality=80)
    rows = "".join(f"<tr><td>{j}</td><td>{html.escape(g['kind'])}</td><td>{g.get('match', '—')}</td><td>{g.get('iou', '')}</td>"
                   f"<td lang=ja>{html.escape(g['text'])}</td><td lang=ja>{html.escape(paddle[s][g['match']-1]['paddle']) if 'match' in g else ''}</td></tr>"
                   for j, g in enumerate(gem[s], 1))
    parts.append(f"<h3>{s}</h3><div style='display:flex;gap:12px;flex-wrap:wrap'><img style='max-width:520px' src='data:image/jpeg;base64,{base64.b64encode(buf.getvalue()).decode()}'>"
                 f"<table><tr><th>G#</th><th>kind</th><th>對到 P#</th><th>IoU</th><th>Gemini</th><th>Paddle</th></tr>{rows}</table></div>")
page = ("<meta charset=utf-8><style>td,th{border:1px solid #ccc;padding:3px 5px;vertical-align:top}table{border-collapse:collapse}</style>"
        + "".join(parts))
open(f"{DRIVE_DIR}/report_gemini_boxes.html", "w").write(page)
display(HTML(page))